# PROJECT 1 · STEP 2 — Hypothesis-oriented EDA

> 본 데이터는 공개 문헌과 공정 메커니즘을 참고하여 프로젝트 검증 목적으로 생성한 synthetic engineering dataset이다.

목표는 ML 정확도가 아니라 `Random vs Pattern`, `Edge`, `Chamber`, `Time Drift`, `Material Lot`, `Parameter Shift`를 순서대로 판별하는 것이다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
die = pd.read_csv(ROOT / 'data/processed/die_level.csv')
shots = pd.read_csv(ROOT / 'data/processed/shot_level.csv', parse_dates=['timestamp'])
summary = json.loads((ROOT / 'results/eda_summary.json').read_text(encoding='utf-8'))
pd.Series(summary, name='value')

## 1. Spatial pattern

Wafer map과 vector map은 `figures/01_worst_shot_void_map.svg`, `02_chip_offset_vector_map.svg`에 저장된다. Edge void가 특정 방위에 집중되고 offset vector가 같은 방향성을 보이는지 확인한다.

In [ ]:
die.groupby('edge_band').agg(void_mean=('void_area_ratio_pct','mean'), offset_p95=('chip_offset_um', lambda s: s.quantile(.95)), n=('die_id','size'))

## 2. Equipment / time / material pattern

Chamber label만으로 결론을 내리지 않고 pump-down, zone range, vent PM age와 EMC genealogy를 분리한다.

In [ ]:
display(shots.groupby('chamber')['edge_void_pct'].agg(['count','mean','std']).sort_values('mean', ascending=False))
display(shots.groupby('emc_lot')['edge_void_pct'].agg(['count','mean','std']).sort_values('mean', ascending=False))
shots[['edge_void_pct','chip_offset_p95_um','pump_down_time_s','zone_range_c','viscosity_index','process_margin_s']].corr().round(2)

## 3. Engineering interpretation

- H1: process margin과 vacuum×zone interaction이 공간 pattern을 함께 설명하는가?
- H2: 특정 chamber와 vent PM age가 trace drift 및 회복을 보이는가?
- H3: suspect EMC lot이 chamber를 넘어 재현되는가?

전체 Observation / Engineering Interpretation / Next Action은 `report/02_eda_findings.md`에 Figure별로 기록한다.

In [ ]:
pd.read_csv(ROOT / 'results/root_cause_evidence.csv')